### Connecting to a runtime

Before you begin, connect to a Python GPU-enabled runtime.

#### Install required dependencies

In [ ]:
!apt-get update
!apt-get install -y iputils-ping
!apt-get install -y traceroute

#### Import required libraries

In [ ]:
from google.colab import files
import torch
import time
import requests
import pandas as pd

#### Confirm that your runtime is GPU-supported

Observe the GPU type, its power draw and its maximum power.

In [ ]:
!nvidia-smi

Next, get the CPU information.

In [ ]:
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread\(s\) per core'
!nproc

Search for the power (TDP) of this CPU model and how many cores and threads the full chip has. Note that your VM only gets a few virtual CPUs (vCPUs, the `nproc` value above) of a much larger chip shared with other users. We will use this later to estimate CPU power.

Estimate the power of your VM's CPUs. `psutil` reports the utilisation of the VM's vCPUs, so the power to multiply it with is the share of the chip that the VM uses, not the whole chip's TDP:

*CPU power of the VM* ≈ TDP of the chip × (vCPUs of the VM ÷ threads of the chip)

Fill in the values you found for your CPU model.

In [ ]:
HOST_TDP_W = 205       # <- TDP of the whole CPU chip (W), from the CPU's specification
HOST_THREADS = 56      # <- hardware threads of the whole chip (cores x threads per core)
import psutil
vcpus = psutil.cpu_count(logical=True)

cpu_power_w = HOST_TDP_W * vcpus / HOST_THREADS
print(f"{vcpus} vCPUs of a {HOST_THREADS}-thread, {HOST_TDP_W} W chip -> about {cpu_power_w:.1f} W at full load")

### 1. GPU start-up cost

The first time a program uses the GPU, the driver has to create a *CUDA context*: load the driver state, reserve GPU memory and bring the GPU out of its low-power state. This costs time (and energy) before any useful work is done. `nvidia-smi` above showed the GPU idling in a low-power state (`P8`).

Run the following code and compare the three times.

In [ ]:
t0 = time.perf_counter()
available = torch.cuda.is_available()
t1 = time.perf_counter()
print(f"torch.cuda.is_available(): {t1 - t0:.5f} s (asks the driver whether a GPU exists; the first call also loads the driver library)")

if available:
    t0 = time.perf_counter()
    torch.zeros(1, device="cuda")      # first allocation creates the CUDA context
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    print(f"First use of the GPU:       {t1 - t0:.5f} s")

    t0 = time.perf_counter()
    torch.zeros(1, device="cuda")
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    print(f"Second use of the GPU:      {t1 - t0:.5f} s")

Run `!nvidia-smi` again. Which performance state (`P0`–`P8`) and power draw does the GPU show now that a program holds a CUDA context? What does that mean for the energy of a GPU that is allocated but not doing any work?

In [ ]:
!nvidia-smi --query-gpu=pstate,power.draw,power.limit --format=csv

### 2. CPU vs. GPU performance

Using a GPU can greatly reduce the execution time of some tasks. Here, we compare CPU and GPU performance on a matrix multiplication task.

Two details matter when timing a GPU:

* **Warm-up:** the first GPU call also loads libraries (cuBLAS), so we do one unmeasured multiplication first.
* **Synchronisation:** GPU calls return *before* the GPU has finished. Without `torch.cuda.synchronize()` you only measure how long it takes to *queue* the work.

In [ ]:
import pynvml
try:
    pynvml.nvmlInit()
    gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
except pynvml.NVMLError:
    gpu_handle = None

def gpu_energy_j():
    """GPU energy counter in joules (counts up since the driver was loaded)."""
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(gpu_handle) / 1000   # millijoules -> joules

n = 5000
reps = {"cpu": 10, "cuda": 100}   # more GPU repetitions so the run lasts several seconds

for name in ["cpu", "cuda"]:
    if name == "cuda" and not torch.cuda.is_available():
        print("GPU unavailable, skipping.")
        continue
    device = torch.device(name)
    x = torch.rand(n, n, device=device)
    flop = 2 * n**3 * reps[name]   # floating-point operations in this run

    y = x @ x                      # warm-up, not measured
    if name == "cuda":
        torch.cuda.synchronize()
        e0 = gpu_energy_j()

    t0 = time.perf_counter()
    for i in range(reps[name]):
        y = x @ x                  # same work every time (x @ x @ x ... would overflow to inf)
    if name == "cuda":
        torch.cuda.synchronize()   # wait until the GPU has really finished
    t1 = time.perf_counter()
    seconds = t1 - t0

    print(f"{name}: {seconds:.3f} s for {flop / 1e12:.1f} TFLOP = {flop / seconds / 1e9:,.0f} GFLOP/s")
    if name == "cpu":
        joules = cpu_power_w * seconds   # assumes all vCPUs fully busy during the run
        print(f"      CPU energy (estimate): {joules:.0f} J = {joules / 3.6e6:.6f} kWh, "
              f"{cpu_power_w:.1f} W, {joules / (flop / 1e9) * 1000:.2f} mJ per GFLOP")
    if name == "cuda":
        joules = gpu_energy_j() - e0
        print(f"      GPU energy: {joules:.0f} J = {joules / 3.6e6:.6f} kWh, "
              f"average {joules / seconds:.1f} W, {joules / (flop / 1e9) * 1000:.2f} mJ per GFLOP")

What is the difference in performance? Write down your observations.

The GPU energy above is **measured** by the GPU's own energy counter. For the CPU there is no counter, so the CPU energy is an **estimate**: `cpu_power_w` × run time, assuming both vCPUs are fully busy. Compare the energy per GFLOP of the CPU and the GPU. What is the GPU's average power compared with its power limit from `nvidia-smi`? How does it compare with your laptop in week 5?

*Optional:* comment out the `torch.cuda.synchronize()` lines and run again. What time do you get for the GPU now, and why is it wrong?

### 3. Colab cloud VM location

Running processes on a cloud instance using a GPU can significantly decerease the computation time. However, the carbon emissions depend largely on the geolocation of the server where the virtual machine (VM) is running.

The geolocation of the Google Cloud VM your notebook is running on can be determined by obtaining its IP address.

Get the external IP address of the Colab server instance.

In [ ]:
!curl -s https://ipinfo.io

### 4. Carbon intensity of cloud compute

First, print again the location of the Colab VM.

In [ ]:
loc = requests.get("https://ipinfo.io").json()
print(f"Colab server location: {loc['city']}, {loc['region']}, {loc['country']}")

Now we look up the carbon intensity of the electricity grid where the VM runs. Google publishes yearly data for every Google Cloud region (no registration needed):

* **Grid carbon intensity** (gCO₂e/kWh): average emissions of the local electricity grid (*location-based*).
* **Google CFE**: the share of each hour's electricity that was carbon-free, counting Google's own clean-energy contracts in that grid.

The next cell picks the region whose location matches the `ipinfo.io` city or state (e.g. Las Vegas → `us-west4`, South Carolina → `us-east1`). If it finds none, or more than one, set `my_region` yourself from the table.

**Colab can put each session in a different region**, so write down the region you got: your results depend on it.

In [ ]:
year = 2025
cfe = pd.read_csv(f"https://raw.githubusercontent.com/GoogleCloudPlatform/region-carbon-info/main/data/yearly/{year}.csv")
cfe.columns = ["region", "location", "google_cfe", "grid_gco2_per_kwh"]

# Match the ipinfo.io city or state with the region's location, e.g. "Las Vegas" or "South Carolina".
places = [loc["city"], loc["region"]]
matches = cfe[cfe["location"].fillna("").apply(
    lambda l: l != "" and any(l == p or l in p or p in l for p in places))]
print(f"Regions matching {', '.join(places)}: {list(matches['region'])}")

my_region = matches["region"].iloc[0] if len(matches) == 1 else None
# my_region = "us-east1"   # <- set it yourself if no single region matched

if my_region not in set(cfe["region"]):
    raise SystemExit("No single region matched: run `cfe` in a new cell to see all regions, "
                     "and set my_region by hand (uncomment the line above).")
row = cfe[cfe["region"] == my_region].iloc[0]
carbon_intensity = row["grid_gco2_per_kwh"]
print(f"{my_region} ({row['location']}), {year}: {carbon_intensity:.0f} gCO2e/kWh, "
      f"{row['google_cfe']:.0%} carbon-free energy")

cfe.sort_values("grid_gco2_per_kwh")

Look at the whole table. Which regions have the lowest and highest grid carbon intensity? How far apart are they? Where is the region closest to Slovenia, and what is its carbon intensity?

These are **yearly averages**. The live mix changes by the hour: open https://app.electricitymaps.com/map and find your VM's region. Which sources are dominant right now, and how does the live value compare with the yearly average? Compare it with your country's carbon intensity. Write down your observations.

For the matrix multiplication example above, calculate the carbon emissions in g CO₂e, using `carbon_intensity`.

### 5. Data transfer speed

Cloud computing that ocurrs thousands of kilometers away is also increasing latency and energy for data transfer.

Therefore, next we are going to measure the data transfer latency and energy consumtion based on the location the data is hosted on.

For measuring data transfer, Colab cannot sniff packets, but we *can* measure effective throughput.

#### 4.1. Download the MNIST dataset to your computer

Open terminal and type

```
wget https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz
```

The output will give you the length of the file in bytes and the time it takes to transfer the file from a remote server to your local machine. Write it down.

#### 4.2 Upload MNIST from your computer to Colab

Now, upload the mnist.npz file from your local machine to Colab and time it.

`files.upload()` prompts you to choose files from your local machine and uploads them to the runtime.

**Caution:** the time below also includes the time you spend in the file dialog. Have the file ready and select it as quickly as possible, repeat it a few times, and keep in mind that the result is an upper bound for the transfer time.

In [ ]:
t0 = time.time()
uploaded = files.upload()
t1 = time.time()

size_mb = sum(len(data) for data in uploaded.values()) / 1e6
print(f"Local->Colab upload: {size_mb:.1f} MB in {t1 - t0:.2f} s = {size_mb / (t1 - t0):.2f} MB/s "
      f"({size_mb * 8 / (t1 - t0):.1f} Mbit/s)")

#### 4.3. Download the same file from Colab back to your machine

In [ ]:
files.download('/content/mnist.npz')

Download time cannot be measured, but can you observe a difference between downloading the same file from its original location using `wget` and from Colab?

4.4. Download MNIST directly to Colab

Now, download the same dataset to Colab using `wget`.
How much time this takes and what is the difference when the file is uploaded from your local machine and from another remote cloud storage?

In [ ]:
!wget -O /dev/null https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz

The output will give you the length of the file in bytes and the time it takes to transfer the file from a remote cloud storage to Colab.

If you were to run a machine learning model on Colab, where would you download the data from? Does it depend on the location where the data is stored?

Now, let's try to estimate the distance to the dataset location.

First, let's use ping to roughly estimate the distance from our Google Cloud VM where Colab is running to the location of the Google Cloud Storage where our MNIST dataset is locates.

CAUTION: Ping will only tell you:
 Packets transmitted and received.

* Round-trip time (RTT)
* The IP address that responded
* Packet loss percentage
* Minimum, maximum, average, and standard deviation of the round-trip time

The IP address you get is not reliable for geolocation because Google Cloud Storage endpoints (including storage.googleapis.com) are served using anycast routing. This means that many servers around the world share the same IP address, your packets are routed to the nearest Google edge location, and the IP does not indicate the physical data center.

However, ping can help you roughly estimate proximity:

* RTT ~1–10 ms - likely the same region
* RTT ~30–80 ms - likely the same continent
* RTT >150 ms - probably cross-continent

In [ ]:
!ping -c 5 www.storage.googleapis.com

What can you conclude based on the round-trip time?

Let's see how many hops exist between Google Cloud VM and Google Cloud Storage where MNIST is stored.

In [ ]:
!traceroute www.storage.googleapis.com

Run the same command on your local machine. The last hop is the hop to the destination (Google Cloud Storage).

What latencies do you observe?

### 6. ML model training energy consumption

Finally, we will investigate the energy consumed when training a ML model on CPU and GPU.

We will measure the execution time and the CPU and GPU utilization and power consumption in order to estimate the carbon emissions.

Install the necessary dependencies first

In [ ]:
!pip install psutil nvidia-ml-py

Load the necessary libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import gc
import numpy as np
import pandas as pd
import os
import psutil
import pynvml

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import metrics

import threading
import time
from time import perf_counter as timer

import warnings
warnings.filterwarnings("ignore")

Resource logger

Examine the class and describe what it does.

In [ ]:
class ResourceLogger:
    """Samples CPU utilisation (psutil) and GPU utilisation and power (NVML) in a background thread."""

    def __init__(self, interval=0.2):
        self.interval = interval
        self.running = False
        self.data = []
        try:
            pynvml.nvmlInit()
            self.gpu = pynvml.nvmlDeviceGetHandleByIndex(0)
        except pynvml.NVMLError:
            self.gpu = None

    def get_gpu_stats(self):
        if self.gpu is None:
            return None, None, None
        util = pynvml.nvmlDeviceGetUtilizationRates(self.gpu)
        power_w = pynvml.nvmlDeviceGetPowerUsage(self.gpu) / 1000   # milliwatts -> watts
        return util.gpu, util.memory, power_w

    def sample(self):
        while self.running:
            cpu = psutil.cpu_percent(interval=None)
            gpu_util, mem_util, power = self.get_gpu_stats()
            self.data.append({
                "timestamp": time.time(),
                "cpu_util": cpu,
                "gpu_util": gpu_util,
                "gpu_mem": mem_util,
                "gpu_power_w": power
            })
            time.sleep(self.interval)

    def start(self):
        self.data = []
        psutil.cpu_percent(interval=None)   # the first call only starts psutil's measurement
        self.running = True
        self.thread = threading.Thread(target=self.sample)
        self.thread.start()

    def stop(self):
        self.running = False
        self.thread.join()

logger = ResourceLogger(interval=0.2)

Download and load the dataset

In [ ]:
!wget -O YearPredictionMSD.txt.zip "https://archive.ics.uci.edu/ml/machine-learning-databases/00203/YearPredictionMSD.txt.zip"

In [ ]:
local_url = "YearPredictionMSD.txt.zip"

print("Loading CSV...")
start = timer()
year = pd.read_csv(local_url, header=None)

load_time = timer() - start
print(f"Dataset load time: {load_time:.2f} s")

x = year.iloc[:, 1:].to_numpy(dtype=np.float32)
y = year.iloc[:, 0].to_numpy(dtype=np.float32)

del year
gc.collect()

The file was already downloaded to the Colab VM in the previous cell. Compare the download time with the load time: what takes longer, transferring 200 MB over the network or unzipping and parsing it? What does that suggest about where to optimise a data pipeline?

Preprocess and prepare the dataset

In [ ]:
# Train-test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

# Normalization
scaler_x = MinMaxScaler()
scaler_y = StandardScaler()

scaler_x.fit(x_train)
x_train = scaler_x.transform(x_train)
x_test = scaler_x.transform(x_test)

scaler_y.fit(y_train.reshape(-1, 1))
y_train = scaler_y.transform(y_train.reshape(-1, 1)).ravel()
y_test = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

PyTorch Linear Regression Model

In [ ]:
class LinearRegressionModel(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = nn.Linear(in_features, 1, bias=True)

    def forward(self, x):
        return self.linear(x)

Helper scripts for energy use calculation

In [ ]:
def gpu_power_to_energy(logs, power_key="gpu_power_w"):
    """
    logs: list of dicts with 'timestamp' and power_key
    Integrates power over time (trapezoidal rule).
    Returns energy_kwh, duration_s
    """
    logs = [l for l in logs if l.get(power_key) is not None]
    if len(logs) < 2:
        return 0.0, 0.0

    energy_j = 0.0
    for i in range(1, len(logs)):
        dt = max(0.0, logs[i]["timestamp"] - logs[i-1]["timestamp"])
        energy_j += 0.5 * (logs[i-1][power_key] + logs[i][power_key]) * dt  # watts * seconds = joules

    duration = logs[-1]["timestamp"] - logs[0]["timestamp"]
    return energy_j / 3_600_000.0, duration

def cpu_util_to_energy(logs, cpu_power_w):
    """
    logs: list of dicts with 'timestamp' and 'cpu_util'
    Estimates CPU energy as cpu_power_w x average utilisation x time.
    Returns energy_kwh, duration_s
    """
    if len(logs) < 2:
        return 0.0, 0.0

    total_area = 0.0
    total_time = 0.0
    for i in range(1, len(logs)):
        dt = max(0.0, logs[i]["timestamp"] - logs[i-1]["timestamp"])
        p0 = logs[i-1].get("cpu_util") or 0.0
        p1 = logs[i].get("cpu_util") or 0.0
        total_area += 0.5 * (p0 + p1) * dt
        total_time += dt

    avg_percent = (total_area / total_time) if total_time > 0 else 0.0
    cpu_energy_j = cpu_power_w * (avg_percent / 100.0) * total_time
    return cpu_energy_j / 3_600_000.0, total_time

The CPU power estimate `cpu_power_w` comes from the beginning of the notebook. If you change it, re-run that cell.

First, measure the **idle** baseline: what the VM and its GPU draw while doing nothing for 20 seconds.

In [ ]:
logger.start()
time.sleep(20)
logger.stop()

idle_cpu_kwh, idle_s = cpu_util_to_energy(logger.data, cpu_power_w)
idle_gpu_kwh, _ = gpu_power_to_energy(logger.data)
idle_cpu_w = idle_cpu_kwh * 3_600_000 / idle_s if idle_s else 0.0
idle_gpu_w = idle_gpu_kwh * 3_600_000 / idle_s if idle_s else 0.0
print(f"Idle: CPU {idle_cpu_w:.1f} W (estimated), GPU {idle_gpu_w:.1f} W (measured)")

Run training on CPU and GPU with utilisation logging, with two ways of feeding the data to the model:

* **`dataloader`**: PyTorch's `DataLoader`, which fetches the training samples one by one in Python and stacks them into batches.
* **`slices`**: shuffle the sample indices once per epoch and take each batch as one slice of the tensors.

The model, batch size and learning rate are the same. The `slices` version runs 100 epochs instead of 10 so that it lasts long enough to measure; compare the **energy per epoch** (in mWh) (its MSE is also lower, because it trains for more epochs). Each combination runs 3 times. Energy is in **Wh** (1 Wh = 0.001 kWh; 1 mWh = 0.001 Wh); *above idle* subtracts the idle power measured above.

`g_co2e` uses your VM's region; `g_co2e_ref` converts the same energy with the Slovenian grid intensity, so you can compare your results with classmates who got a different region.

In [ ]:
def train(device, loading="dataloader", epochs=10, batch_size=2048):
    X_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    Y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_test = torch.tensor(x_test, dtype=torch.float32).to(device)
    Y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    model = LinearRegressionModel(X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True)

    def batches():
        if loading == "dataloader":
            yield from loader                      # one sample at a time, then stacked
        else:
            perm = torch.randperm(X_train.shape[0], device=device)
            for start in range(0, X_train.shape[0], batch_size):
                idx = perm[start:start + batch_size]
                yield X_train[idx], Y_train[idx]   # one slice per batch

    logger.start()
    start = timer()
    for epoch in range(epochs):
        for xb, yb in batches():
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    if device == "cuda":
        torch.cuda.synchronize()
    train_time = timer() - start
    logger.stop()

    model.eval()
    with torch.no_grad():
        mse = metrics.mean_squared_error(Y_test.cpu().numpy(), model(X_test).cpu().numpy())

    cpu_kwh, _ = cpu_util_to_energy(logger.data, cpu_power_w)
    gpu_kwh, _ = gpu_power_to_energy(logger.data)
    if len(logger.data) < 10:
        print(f"  only {len(logger.data)} samples: run more epochs for a reliable energy value")
    gpu_util = [d["gpu_util"] for d in logger.data if d["gpu_util"] is not None]

    del model, X_train, Y_train, X_test, Y_test, loader
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return {"device": device, "loading": loading, "epochs": epochs, "time_s": train_time, "mse": mse,
            "cpu_util_%": np.mean([d["cpu_util"] for d in logger.data]) if logger.data else 0.0,
            "gpu_util_%": np.mean(gpu_util) if gpu_util else 0.0,
            "cpu_wh": cpu_kwh * 1000, "gpu_wh": gpu_kwh * 1000}

REFERENCE_INTENSITY = 250   # <- gCO2e/kWh of the Slovenian grid; use the value you found in week 2

devices = ["cpu", "cuda"] if torch.cuda.is_available() else ["cpu"]
setups = [("dataloader", 10), ("slices", 100)]
runs = []
for run in range(3):
    for loading, epochs in setups:
        for device in devices:            # alternate between setups in every round
            print(f"Run {run + 1}: {device}, {loading}, {epochs} epochs...")
            runs.append(train(device, loading, epochs))

runs = pd.DataFrame(runs)
runs["total_wh"] = runs["cpu_wh"] + runs["gpu_wh"]
runs["above_idle_wh"] = runs["total_wh"] - (idle_cpu_w + idle_gpu_w) * runs["time_s"] / 3600
runs["mwh_per_epoch"] = runs["total_wh"] * 1000 / runs["epochs"]
runs["g_co2e"] = runs["total_wh"] / 1000 * carbon_intensity                 # your VM's region
runs["g_co2e_ref"] = runs["total_wh"] / 1000 * REFERENCE_INTENSITY          # same energy in Slovenia
runs.round(4)

In [ ]:
cols = ["time_s", "gpu_util_%", "cpu_wh", "gpu_wh", "total_wh", "above_idle_wh", "mwh_per_epoch", "g_co2e", "g_co2e_ref"]
summary = runs.groupby(["loading", "device"])[cols].agg(["mean", "std"])
summary.round(4)

Compare the results and comment on them:

* With the `dataloader` version, was the GPU faster than the CPU? What was the GPU utilisation? Where does the time go: in the computation, or in Python handling one sample at a time?
* How much did the `slices` version change the time and the energy per epoch, on the CPU and on the GPU? Which device wins now?
* Look at `gpu_wh` in the **CPU** runs and compare it with the idle measurement. How much of the total energy is a GPU that is allocated but not used? Compare `total_wh` with `above_idle_wh`: which one would you report, and why?
* `g_co2e` uses the carbon intensity of your VM's region. How would it change in the lowest-carbon region in the table?
* *Optional:* the model is so small that even the `slices` version keeps the GPU mostly idle. Run `train("cuda", "slices", 100, batch_size=65536)` and `train("cpu", "slices", 100, batch_size=65536)`. How do GPU utilisation, time and energy per epoch change? (Larger batches also change how the model learns: compare the MSE.)